# Intelligent Crime Detective Platform — Person B Analytics Notebook
## Day 2: Dataset Cleaning & Feature Engineering

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 2 of 10-Day Development Plan  

### Core Governance & Methodological Principles for Day 2
1. **Raw Data Immutability:** Raw datasets in `data/raw/` are immutable primary sources and are never altered.
2. **No Premature ML:** In accordance with the project roadmap, no model training, PCA, LWR, ID3, Naive Bayes, or k-NN algorithms are executed today.
3. **Target Leakage Prevention:** `disposition` and its derived indicator `is_solved` represent ground-truth investigative outcomes and are strictly isolated as target variables, never input features.
4. **Semantic Preservation:** Missing coordinates and unknown ages are never filled with `0`. Infant age `0` is strictly preserved as valid.
5. **Clean Processed Outputs:** Cleaned datasets with documented lineage are saved under `data/processed/`.

## 1. Environment Setup

Initialize runtime environment, configure paths, and import foundational libraries and internal utilities.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUTS_DIR
from src.data_inspection import inspect_dataset
from src.data_cleaning import (
    load_csv, strip_whitespace, normalize_categorical,
    detect_duplicates, report_missing_values, parse_dates_safely,
    coerce_numeric, validate_coordinates, identify_total_rows
)
from src.data_quality import dataset_summary, duplicate_report, profile_columns, generate_quality_report, missing_value_report
from src.feature_engineering import (
    extract_temporal_features, create_target_solvability,
    flag_target_leakage_columns, standardize_tamil_nadu_districts
)
from src.run_cleaning_pipeline import run_all_cleaning

print(f"Project Root: {PROJECT_ROOT}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

Project Root: /home/Dharsit/ML-Project/Crimora
Pandas Version: 3.0.5
NumPy Version: 2.5.3


## 2. Dataset Discovery

Programmatically discover and verify the physical files present in `data/raw/`.

In [ ]:
raw_files = sorted(list(RAW_DATA_DIR.glob("*.csv")))
print(f"Discovered {len(raw_files)} CSV datasets in {RAW_DATA_DIR}:")

discovery_records = []
for f in raw_files:
    size_kb = round(f.stat().st_size / 1024, 2)
    discovery_records.append({
        "Filename": f.name,
        "File Size (KB)": size_kb,
        "File Size (MB)": round(size_kb / 1024, 2),
    })

df_discovery = pd.DataFrame(discovery_records)
display(df_discovery)

## 3. Dataset Inventory

Define the analytical scope, target domain, and planned Person B modeling modules for each discovered dataset.

In [ ]:
inventory_data = [
    {
        "Filename": "homicide-data.csv",
        "Target Domain": "US Major City Homicides (50 cities)",
        "Target / Outcome Column": "disposition (Closed by arrest vs others)",
        "Planned Person B Modules": "Solvability Estimation (ID3, Naive Bayes, k-NN), Geographic Profiling (LWR)",
    },
    {
        "Filename": "dstrIPC_1_2014.csv",
        "Target Domain": "NCRB National District IPC Crime Records (2014)",
        "Target / Outcome Column": "Total Cognizable IPC crimes (Aggregate Benchmark)",
        "Planned Person B Modules": "Principal Component Analysis (PCA), Crime Offense Pattern Profiling",
    },
    {
        "Filename": "TN-2020-2022-total.csv",
        "Target Domain": "Tamil Nadu Multi-Year Cognizable Crime Totals (2020-2022)",
        "Target / Outcome Column": "Rate of Cognizable crime (IPC+SLL)",
        "Planned Person B Modules": "Longitudinal Trend Analysis, District Growth Comparison, Streamlit UI",
    },
    {
        "Filename": "TN-murder-2023.csv",
        "Target Domain": "Tamil Nadu Violent & Negligent Fatalities (2023)",
        "Target / Outcome Column": "Murder - Rate / Incidence",
        "Planned Person B Modules": "Tamil Nadu Geospatial Analytics, Folium Interactive Map",
    },
]

df_inventory = pd.DataFrame(inventory_data)
display(df_inventory)

## 4. Schema Inspection

Inspect the native dimensions, column headers, and data types across each source dataset.

In [ ]:
schema_summaries = []
loaded_raw = {}

for f in raw_files:
    enc = "latin-1" if "homicide" in f.name else "utf-8"
    df_temp = load_csv(f, encodings_to_try=[enc, "latin-1", "utf-8"])
    loaded_raw[f.name] = df_temp
    
    schema_summaries.append({
        "Dataset": f.name,
        "Rows": len(df_temp),
        "Columns": len(df_temp.columns),
        "Numeric Cols": len(df_temp.select_dtypes(include=[np.number]).columns),
        "String/Object Cols": len(df_temp.select_dtypes(include=["object", "string"]).columns),
    })

display(pd.DataFrame(schema_summaries))

## 5. Data Quality Analysis

Generate comprehensive quality audits across the raw data assets.

In [ ]:
quality_metrics = []
for name, df_temp in loaded_raw.items():
    summary = dataset_summary(df_temp, dataset_name=name)
    quality_metrics.append(summary)

display(pd.DataFrame(quality_metrics))

## 6. Missing Value Analysis

Audit missing value counts, percentages, and semantic causes across all fields.

In [ ]:
for name, df_temp in loaded_raw.items():
    rep = missing_value_report(df_temp)
    missing_cols = rep[rep["missing_count"] > 0]
    print(f"=== Missing Values in {name} ===")
    if len(missing_cols) == 0:
        print("  -> No missing values found in native DataFrame.")
    else:
        display(missing_cols)

print("\nSemantic Missingness Audit:")
print("1. homicide-data.csv: lat & lon have 60 nulls (0.11%). victim_age has 2,999 'Unknown' entries.")
print("2. TN-2020-2022-total.csv: Avadi and Tambaram have 'N/C' (Not Created) in 2020 & 2021.")
print("3. TN-murder-2023.csv: Specialized units (Railways, Cyber Cell) have '-' rates.")

## 7. Duplicate Analysis

Examine exact duplicate rows and test candidate primary keys for collisions.

In [ ]:
dup_results = []
for name, df_temp in loaded_raw.items():
    id_col = "uid" if "uid" in df_temp.columns else ("Sl No" if "Sl No" in df_temp.columns else None)
    d_rep = duplicate_report(df_temp, id_col=id_col)
    dup_results.append({
        "Dataset": name,
        "Exact Duplicate Rows": d_rep["exact_duplicate_rows"],
        "ID Column": d_rep["id_column"],
        "Duplicate IDs Count": d_rep["duplicate_ids_count"],
    })

display(pd.DataFrame(dup_results))

## 8. Categorical Feature Analysis

Profile categorical values, label cardinality, and distributions.

In [ ]:
df_hom = loaded_raw["homicide-data.csv"]
print("Victim Sex Distribution:")
display(df_hom["victim_sex"].value_counts(dropna=False).to_frame(name="Count"))

print("\nVictim Race Distribution:")
display(df_hom["victim_race"].value_counts(dropna=False).to_frame(name="Count"))

print("\nCase Disposition (Ground-Truth Clearance):")
display(df_hom["disposition"].value_counts(dropna=False).to_frame(name="Count"))

## 9. Numerical Feature Analysis

Inspect numerical distributions, boundary limits, and anomalies.

In [ ]:
# Inspect victim age distribution in homicide data
numeric_age = coerce_numeric(df_hom["victim_age"], sentinel_strings=["Unknown"])
print("Victim Age Statistics (excluding 'Unknown'):")
display(numeric_age.describe().to_frame(name="Victim Age Distribution"))

print(f"Infant records (Age == 0): {(numeric_age == 0).sum()} cases (preserved as valid)")
print(f"Unknown age records: {numeric_age.isnull().sum()} cases (represented as NaN)")

## 10. Geographic Feature Analysis

Validate latitude/longitude coordinate bounds and analyze district jurisdiction nomenclature.

In [ ]:
# Validate coordinates in homicide data
valid_coords = validate_coordinates(df_hom, lat_col="lat", lon_col="lon")
print("Coordinate Validity Audit:")
print(f"  Valid coordinates: {valid_coords.sum():,} ({valid_coords.mean()*100:.2f}%)")
print(f"  Missing/Invalid coordinates: {(~valid_coords).sum()} (0.11%)")
print(f"  Latitude range (valid): [{df_hom.loc[valid_coords, 'lat'].min():.4f}, {df_hom.loc[valid_coords, 'lat'].max():.4f}]")
print(f"  Longitude range (valid): [{df_hom.loc[valid_coords, 'lon'].min():.4f}, {df_hom.loc[valid_coords, 'lon'].max():.4f}]")

# Check Tamil Nadu district transliteration differences
df_tn_tot = loaded_raw["TN-2020-2022-total.csv"]
df_tn_mrd = loaded_raw["TN-murder-2023.csv"]
print("\nTamil Nadu District Jurisdictions in 2020-2022 Totals:", len(df_tn_tot))
print("Tamil Nadu District Jurisdictions in 2023 Murder:", len(df_tn_mrd))
new_dist = set(df_tn_mrd["Districts/City"]) - set(df_tn_tot["Districts"])
print("New district appearing in 2023:", new_dist)

## 11. Temporal Feature Analysis

Analyze reporting dates, handle entry typos safely, and extract calendar features.

In [ ]:
dates_raw = df_hom["reported_date"].astype(str)
print("Date string length distribution:")
display(dates_raw.str.len().value_counts().to_frame(name="Count"))

malformed = df_hom[dates_raw.str.len() != 8]
print(f"Malformed date records count: {len(malformed)}")
display(malformed[["uid", "reported_date", "city", "state"]])

dates_parsed = parse_dates_safely(df_hom["reported_date"], format="%Y%m%d", errors="coerce")
print(f"Successfully parsed dates: {dates_parsed.notnull().sum():,}")
print(f"NaT dates (safely coerced): {dates_parsed.isnull().sum()}")

## 12. Cleaning Decisions

Document the explicit, auditable rationale behind all transformations performed.

In [ ]:
decisions = [
    {
        "Aspect": "Missing Coordinates (homicide-data)",
        "Action": "Do not fill with 0. Flag via valid_coords and create homicide_spatial_clean.csv.",
        "Rationale": "Filling coordinates with 0 creates fictitious incidents at (0, 0) Off Africa, corrupting GIS maps and LWR."
    },
    {
        "Aspect": "Unknown Victim Age (homicide-data)",
        "Action": "Coerce 'Unknown' to NaN. Preserve age 0.",
        "Rationale": "0 is a valid numerical age for infants (<1 year old). Imputing Unknown with 0 introduces profound bias."
    },
    {
        "Aspect": "Malformed Dates (homicide-data)",
        "Action": "Safely coerce 9-digit integers to NaT while preserving raw reported_date.",
        "Rationale": "Prevents pipeline runtime crashes while maintaining data integrity."
    },
    {
        "Aspect": "Not Created 'N/C' (TN-2020-2022-total)",
        "Action": "Convert 'N/C' to NaN in Avadi and Tambaram for 2020/2021.",
        "Rationale": "Districts created in late 2021 lack separate historical figures. Conversion allows numeric calculations."
    },
    {
        "Aspect": "Special Police Units Rates '-' (TN-murder-2023)",
        "Action": "Convert '-' to NaN in rate columns.",
        "Rationale": "Railways and Cyber Cell lack residential population bases, making per-lakh rates undefined."
    },
    {
        "Aspect": "Aggregate Summary Rows (NCRB & TN)",
        "Action": "Add is_total_row boolean flag.",
        "Rationale": "Prevents double-counting state crime totals during district-level statistical modeling."
    },
    {
        "Aspect": "Target Variable & Leakage",
        "Action": "Derive is_solved (1=Closed by arrest) and label strictly as TARGET ONLY.",
        "Rationale": "Case disposition is a post-investigation outcome; must never be supplied to input features (X)."
    },
]

display(pd.DataFrame(decisions))

## 13. Clean Dataset Generation

Execute the end-to-end data cleaning pipeline and serialize cleaned datasets into `data/processed/`.

In [ ]:
# Execute the modular pipeline runner
processed_outputs = run_all_cleaning()

processed_summary = []
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    processed_summary.append({
        "Dataset Identifier": key,
        "Output File": path.name,
        "Rows": len(df_p),
        "Columns": len(df_p.columns),
        "Size (KB)": round(path.stat().st_size / 1024, 2),
    })

display(pd.DataFrame(processed_summary))

## 14. Feature Dictionary

Preview the structured feature dictionary documented under `outputs/person_b_feature_dictionary.md`.

In [ ]:
feat_dict_path = OUTPUTS_DIR / "person_b_feature_dictionary.md"
if feat_dict_path.exists():
    print(f"Feature Dictionary successfully established at: {feat_dict_path}")
    with open(feat_dict_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"Total lines of documentation: {len(lines)}")
    # Print first 35 lines preview
    print("".join(lines[:35]))
else:
    print("Feature dictionary not found!")

## 15. Final Data Quality Report

Verify the data health and integrity of all processed outputs.

In [ ]:
print("=== FINAL PROCESSED DATA QUALITY VERIFICATION ===")
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    print(f"\n--- {key} ({path.name}) ---")
    print(f"Rows: {len(df_p):,}, Columns: {len(df_p.columns)}")
    print(f"Duplicate Rows: {df_p.duplicated().sum()}")
    null_counts = df_p.isnull().sum()
    active_nulls = null_counts[null_counts > 0]
    if len(active_nulls) > 0:
        print("Null counts per column:")
        for col, cnt in active_nulls.items():
            print(f"  {col}: {cnt} ({cnt/len(df_p)*100:.2f}%)")
    else:
        print("  Zero null cells detected.")

print("\nDay 2 Dataset Cleaning & Feature Engineering successfully completed!")

# Day 3 — PCA and Tamil Nadu Crime Analytics

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 3 of 10-Day Development Plan  

### Objectives for Day 3:
1. **PCA Macro-Level Analytics Pipeline:** Feature selection, target leakage prevention, nominal one-hot encoding, non-zero median imputation, standardized scaling, PCA training, explained variance, and component loadings.
2. **Tamil Nadu Crime Analytics Module:** District-level aggregation, multi-year longitudinal trend analysis, category breakdowns, and commissionerate comparisons.
3. **Publication-Grade Visualizations:** Four diagnostic PCA plots and four Tamil Nadu analytical charts.
4. **Model Bundle Serialization & Reproducibility:** Persisting full deployable preprocessor and PCA pipeline in `models/pca_model.joblib`.

## 1. Dataset Loading

Load the cleaned datasets produced during Day 2 from `data/processed/` using the modular project paths.

In [ ]:
from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, MODELS_DIR
import pandas as pd
import numpy as np

datasets = {}
for p in sorted(PROCESSED_DATA_DIR.glob("*.csv")):
    df_t = pd.read_csv(p)
    datasets[p.name] = df_t
    print(f"Loaded: {p.name:<35} | Shape: {str(df_t.shape):<12} | Memory: {round(p.stat().st_size/1024, 1):>7} KB")

## 2. Processed Dataset Inspection

Inspect column schemas, data types, null proportions, and governance markers across all processed files.

In [ ]:
from src.feature_engineering import identify_feature_types

inspection_rows = []
for name, df_curr in datasets.items():
    roles = identify_feature_types(df_curr)
    inspection_rows.append({
        "Dataset": name,
        "Total Rows": len(df_curr),
        "Total Cols": len(df_curr.columns),
        "Numeric Cols": len(roles["numerical"]),
        "Categorical Cols": len(roles["categorical"]),
        "Temporal Cols": len(roles["temporal"]),
        "Geographic Cols": len(roles["geographic"]),
        "Target/Leakage Cols": len(roles["target"]),
        "Identifier Cols": len(roles["identifier"]),
    })

df_inspect = pd.DataFrame(inspection_rows)
display(df_inspect)

## 3. Feature Selection

Select analytical features for macro-level incident PCA modeling while strictly eliminating target outcome labels (`is_solved`, `disposition`), PII, raw coordinates, and order identifiers.

In [ ]:
from src.pca_analysis import generate_feature_selection_table

feat_sel_df = generate_feature_selection_table()
print(f"Total Columns Audited: {len(feat_sel_df)}")
print(f"Features Selected for PCA: {feat_sel_df['selected'].sum()}")
print(f"Features Excluded: {(~feat_sel_df['selected']).sum()}")

display(feat_sel_df)

## 4. Feature-Type Identification

Partition selected modeling features into numeric and nominal categorical groupings.

In [ ]:
selected_features = feat_sel_df[feat_sel_df["selected"]]
numeric_cols = selected_features[selected_features["role"].str.contains("numerical")]["column"].tolist()
categorical_cols = selected_features[selected_features["role"].str.contains("categorical")]["column"].tolist()

print(f"Continuous / Numerical Features ({len(numeric_cols)}): {numeric_cols}")
print(f"Nominal Categorical Features ({len(categorical_cols)}): {categorical_cols}")

## 5. Categorical Encoding

Validate nominal categorical encoding using `OneHotEncoder(handle_unknown='ignore')` to ensure safe handling of unseen categories without inducing artificial ordinality.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

df_homicide = datasets["homicide_clean.csv"]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_encoded = ohe.fit_transform(df_homicide[categorical_cols])
ohe_names = ohe.get_feature_names_out(categorical_cols)

print(f"Categorical Input Dimensions: {len(categorical_cols)}")
print(f"One-Hot Encoded Dimensions: {cat_encoded.shape[1]}")
print(f"Sample Encoded Feature Names: {list(ohe_names[:10])}")

## 6. Numerical Preprocessing

Impute continuous demographic missing values using median imputation (`SimpleImputer(strategy='median')`). Infant age 0 is strictly preserved as valid and never corrupted with zero-filled missingness.

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
num_imputed = num_imputer.fit_transform(df_homicide[numeric_cols])

print("Imputation Statistics:")
for col, stat in zip(numeric_cols, num_imputer.statistics_):
    nulls = df_homicide[col].isnull().sum()
    print(f"  - {col:<20}: median={stat:<6.1f} | missing_imputed={nulls}")

print(f"Infant records with age == 0.0: {(df_homicide['victim_age_clean'] == 0.0).sum()}")

## 7. Scaling

Standardize features with `StandardScaler` to achieve zero mean and unit variance prior to eigendecomposition.

In [ ]:
from sklearn.preprocessing import StandardScaler

combined_matrix = np.hstack([num_imputed, cat_encoded])
scaler = StandardScaler()
scaled_matrix = scaler.fit_transform(combined_matrix)

print(f"Combined Matrix Shape: {scaled_matrix.shape}")
print(f"Global Mean Across Transformed Features: {scaled_matrix.mean():.6f} (near 0)")
print(f"Global Std Across Transformed Features:  {scaled_matrix.std():.6f} (near 1)")

## 8. PCA Training

Train the PCA decomposition pipeline and determine the optimal component cutoff.

In [ ]:
from src.pca_analysis import fit_pca_analysis

pca_results = fit_pca_analysis(
    df=df_homicide,
    numeric_features=numeric_cols,
    categorical_features=categorical_cols,
    variance_threshold=0.85,
)

print(f"Recommended Principal Components: {pca_results['recommended_k']}")
print(f"Transformed Dimension Count:       {len(pca_results['transformed_feature_names'])}")
print(f"Cumulative Explained Variance:     {pca_results['explained_variance_df'].iloc[pca_results['recommended_k']-1]['cumulative_explained_variance']*100:.2f}%")

## 9. Explained Variance Analysis

Audit individual explained variance ratios and verify monotonic cumulative variance growth.

In [ ]:
exp_df = pca_results["explained_variance_df"]
display(exp_df.head(15))

cum_var = exp_df["cumulative_explained_variance"].values
is_monotonic = np.all(np.diff(cum_var) >= 0)
print(f"Monotonic Cumulative Growth Verified: {is_monotonic}")

## 10. PCA Component Analysis

Inspect the component loadings matrix to interpret the principal drivers of macro crime variance.

In [ ]:
loadings_df = pca_results["loadings_df"]
print("Top 5 Positive Loadings for PC1:")
display(loadings_df["PC1"].sort_values(ascending=False).head(5))

print("Top 5 Negative Loadings for PC1:")
display(loadings_df["PC1"].sort_values().head(5))

print("Top 5 Positive Loadings for PC2:")
display(loadings_df["PC2"].sort_values(ascending=False).head(5))

## 11. PCA Visualization

Verify that the four publication-quality diagnostic PCA plots exist on disk.

In [ ]:
pca_plots = [
    OUTPUTS_DIR / "pca" / "explained_variance_by_component.png",
    OUTPUTS_DIR / "pca" / "cumulative_explained_variance.png",
    OUTPUTS_DIR / "pca" / "pca_2d_projection.png",
    OUTPUTS_DIR / "pca" / "pca_feature_contributions.png",
]

for p in pca_plots:
    assert p.exists(), f"Plot missing: {p}"
    print(f"Verified Plot Artifact: {p.name:<35} | Size: {round(p.stat().st_size/1024, 1):>6} KB")

## 12. Tamil Nadu Dataset Analysis

Ingest and inspect Tamil Nadu state and district-level crime datasets.

In [ ]:
from src.tn_analytics import load_tn_datasets

df_tn_total, df_tn_murder, df_tn_ipc = load_tn_datasets()
print(f"Tamil Nadu Multi-Year Totals Shape: {df_tn_total.shape}")
print(f"Tamil Nadu 2023 Fatalities Shape:  {df_tn_murder.shape}")
print(f"Tamil Nadu 2014 IPC Baseline Shape: {df_tn_ipc.shape}")

## 13. District Analysis

Generate aggregated district-level summaries, rank jurisdictions, and ensure no false rates are calculated for population-less units.

In [ ]:
from src.tn_analytics import build_district_summary, rank_districts, compare_districts, compute_basic_statistics

district_summary = build_district_summary(df_tn_total, df_tn_murder)
print(f"Total Standardized Districts Summarized: {len(district_summary)}")

print("\n--- Top 10 Jurisdictions by 2022 Total Crime Count ---")
display(rank_districts(district_summary, "crime_count_2022", top_n=10))

print("\n--- Top 10 Jurisdictions by 2023 Violent Fatalities ---")
display(rank_districts(district_summary, "violent_fatalities_total_2023", top_n=10))

print("\n--- Descriptive Statistics Across Districts ---")
display(compute_basic_statistics(district_summary))

## 14. Yearly Crime Trends

Examine multi-year longitudinal crime trends (2020-2022 and 2014 benchmark) and calculate year-over-year mathematical growth without claiming causation.

In [ ]:
from src.tn_analytics import build_yearly_trends

yearly_trends = build_yearly_trends(df_tn_total, df_tn_ipc)
display(yearly_trends)

print("\nResponsible Analytics Note: Trends reflect historical administrative reporting cycles. Pandemics and policy enforcements directly influenced reporting patterns without establishing intrinsic behavioural causation.")

## 15. Tamil Nadu Visualizations

Verify that the four publication-quality Tamil Nadu analytical plots exist on disk.

In [ ]:
tn_plots = [
    OUTPUTS_DIR / "tamil_nadu" / "district_crime_distribution.png",
    OUTPUTS_DIR / "tamil_nadu" / "yearly_crime_trends.png",
    OUTPUTS_DIR / "tamil_nadu" / "crime_category_distribution.png",
    OUTPUTS_DIR / "tamil_nadu" / "district_comparison.png",
]

for p in tn_plots:
    assert p.exists(), f"Plot missing: {p}"
    print(f"Verified Plot Artifact: {p.name:<35} | Size: {round(p.stat().st_size/1024, 1):>6} KB")

## 16. Output Generation

Execute end-to-end pipeline runners and verify the serialization of all Day 3 file artifacts.

In [ ]:
from src.pca_analysis import run_pca_pipeline
from src.tn_analytics import run_tn_pipeline

pca_meta = run_pca_pipeline()
tn_meta = run_tn_pipeline()

print("PCA Execution Summary:")
for k, v in pca_meta.items():
    print(f"  {k}: {v}")

print("\nTamil Nadu Analytics Summary:")
for k, v in tn_meta.items():
    print(f"  {k}: {v}")

## 17. Validation

Execute comprehensive validation checks to confirm numerical consistency, target leakage isolation, model reloadability, and raw data immutability.

In [ ]:
from src.pca_analysis import load_pca_bundle, transform_new_data
from src.config import RAW_DATA_DIR

# 1. Model Reload Test
model_path = MODELS_DIR / "pca_model.joblib"
bundle = load_pca_bundle(model_path)
sample_holdout = df_homicide.sample(20, random_state=99)
trans_holdout = transform_new_data(bundle, sample_holdout)
assert trans_holdout.shape == (20, bundle["recommended_k"])
print("Validation 1: Saved PCA bundle reloads and transforms holdout data successfully.")

# 2. Target Leakage Prevention Check
feature_names_in = bundle["input_features"]
for leakage_kw in ["is_solved", "disposition", "outcome", "arrest"]:
    assert not any(leakage_kw in col.lower() for col in feature_names_in), f"Leakage found: {leakage_kw}"
print("Validation 2: Zero target leakage detected in feature selection.")

# 3. Cumulative Explained Variance Validity
cum_var = exp_df["cumulative_explained_variance"].values
assert np.all(np.diff(cum_var) >= 0), "Cumulative variance must be monotonically increasing."
assert cum_var[-1] <= 1.0001, "Cumulative variance cannot exceed 1.0."
print("Validation 3: Cumulative explained variance is valid and monotonically increasing.")

# 4. District Aggregations & Rate Denominators
ds = pd.read_csv(OUTPUTS_DIR / "tamil_nadu" / "district_summary.csv")
assert "TOTAL DISTRICT(S)" not in ds["district"].values, "Total row must be filtered."
for num_col in ds.select_dtypes(include=[np.number]).columns:
    assert (ds[num_col].dropna() >= 0).all(), f"Negative values in {num_col}"
print("Validation 4: District summaries exclude aggregate rows and have non-negative crime counts.")

# 5. Raw Data Immutability
for rf in RAW_DATA_DIR.glob("*.csv"):
    assert rf.stat().st_mtime < model_path.stat().st_mtime, f"Raw file modified! {rf.name}"
print("Validation 5: Raw datasets in data/raw/ remain strictly unmodified.")

print("\nALL DAY 3 VALIDATION CHECKS PASSED!")

# Day 4 — Geographic Profiling and LWR

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 4 of 10-Day Development Plan  

### Core Governance & Methodological Principles for Day 4
1. **Raw Data Immutability:** Raw datasets in `data/raw/` are immutable primary sources and are never modified.
2. **Defensible Geospatial Modeling:** Spatial crime occurrences are discrete spatial point patterns. Modeling spatial concentration utilizes distance-weighted spatial intensity estimation (kernel spatial profiling), with Locally Weighted Regression (LWR / LOESS) formulated for continuous spatial responses.
3. **Semantic Integrity:** Missing coordinates are never replaced with `(0, 0)` or synthetic coordinates.
4. **PII Isolation:** Personally identifying information (`victim_first`, `victim_last`) is strictly excluded from all analytical geospatial outputs.
5. **Responsible Decision Support:** Analytical outputs represent historical crime concentration and geographic activity-area estimates for strategic resource allocation. They **never** claim to identify an individual offender, an offender's residence, guilt, or causality.

## 1. Geographic Dataset Discovery

Inspect processed datasets in `data/processed/` to identify valid geographic point coordinate fields (`lat`, `lon`), spatial administrative boundaries (`city`, `state`, `district`), and temporal fields (`reported_date_clean`, `reported_year`).

In [1]:
from pathlib import Path
import pandas as pd
from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, GEOGRAPHIC_OUTPUTS_DIR
from src.geo_utils import find_coordinate_columns

print("=== INSPECTING PROCESSED DATASETS FOR GEOGRAPHIC COORDINATES ===")
candidate_files = sorted(PROCESSED_DATA_DIR.glob("*.csv"))

geo_inventory = []
for f in candidate_files:
    sample_df = pd.read_csv(f, nrows=5)
    lat_c, lon_c = find_coordinate_columns(sample_df)
    has_coords = (lat_c is not None and lon_c is not None)
    geo_inventory.append({
        "dataset": f.name,
        "lat_column": lat_c,
        "lon_column": lon_c,
        "has_point_coordinates": has_coords,
        "total_columns": len(sample_df.columns),
    })

inventory_df = pd.DataFrame(geo_inventory)
print(inventory_df.to_string(index=False))
print("\nFinding: 'homicide_clean.csv' and 'homicide_spatial_clean.csv' are the only datasets with incident point coordinates (lat, lon).")

=== INSPECTING PROCESSED DATASETS FOR GEOGRAPHIC COORDINATES ===
                           dataset                                       lat_column lon_column  has_point_coordinates  total_columns
           dstr_ipc_2014_clean.csv Counterfeit Offences related to Counterfeit Coin        NaN                  False             92
                homicide_clean.csv                                              lat        lon                   True             22
        homicide_spatial_clean.csv                                              lat        lon                   True             22
tn_crime_total_2020_2022_clean.csv                                              NaN        NaN                  False              9
          tn_murder_2023_clean.csv                                              NaN        NaN                  False             13

Finding: 'homicide_clean.csv' and 'homicide_spatial_clean.csv' are the only datasets with incident point coordinates (lat, lon).


## 2. Coordinate Validation

Enforce physical spherical coordinate limits:
- Latitude: $[-90.0, 90.0]$
- Longitude: $[-180.0, 180.0]$

Records with missing, null, or out-of-bounds coordinates are safely segregated into an audit table. Under no circumstances are missing coordinates replaced with `0.0` or synthetic locations.

In [2]:
from src.geo_utils import (
    validate_latitude, validate_longitude,
    filter_valid_coordinates, extract_missing_coordinates
)

hom_clean_path = PROCESSED_DATA_DIR / "homicide_clean.csv"
df_hom = pd.read_csv(hom_clean_path)

total_rows = len(df_hom)
valid_mask = validate_latitude(df_hom["lat"]) & validate_longitude(df_hom["lon"])
valid_count = int(valid_mask.sum())
missing_count = total_rows - valid_count

print(f"Total Records: {total_rows:,}")
print(f"Valid Coordinates: {valid_count:,} ({valid_count / total_rows * 100:.2f}%)")
print(f"Missing/Invalid Coordinates: {missing_count:,} ({missing_count / total_rows * 100:.2f}%)")

# Coordinate range verification
valid_df = filter_valid_coordinates(df_hom, "lat", "lon")
missing_df = extract_missing_coordinates(df_hom, "lat", "lon")

print(f"Latitude Range:  [{valid_df['lat'].min():.6f}, {valid_df['lat'].max():.6f}]")
print(f"Longitude Range: [{valid_df['lon'].min():.6f}, {valid_df['lon'].max():.6f}]")
print(f"Missing coordinates preserved separately: {len(missing_df)} rows.")

Total Records: 52,179
Valid Coordinates: 52,119 (99.89%)
Missing/Invalid Coordinates: 60 (0.11%)
Latitude Range:  [25.725214, 45.051190]
Longitude Range: [-122.507779, -71.011519]
Missing coordinates preserved separately: 60 rows.


## 3. Spatial Data Preparation

Export a sanitized, privacy-preserving coordinate dataset containing only necessary analytical features (`uid`, `city`, `state`, `lat`, `lon`, `reported_date_clean`, `reported_year`, `reported_month`, `is_solved`, `disposition`), strictly omitting personal identifying information (`victim_first`, `victim_last`).

In [3]:
from src.geo_utils import export_clean_crime_coordinates

coords_export_path = GEOGRAPHIC_OUTPUTS_DIR / "crime_coordinates.csv"
clean_coords_df = export_clean_crime_coordinates(
    df=df_hom,
    output_path=coords_export_path,
    lat_col="lat",
    lon_col="lon",
)

print(f"Exported clean coordinates to: {coords_export_path}")
print(f"Total exported rows: {len(clean_coords_df):,}")
print(f"Columns: {list(clean_coords_df.columns)}")
assert "victim_first" not in clean_coords_df.columns
assert "victim_last" not in clean_coords_df.columns
print("Privacy verification: Zero PII fields present in export.")

Exported clean coordinates to: /home/Dharsit/ML-Project/Crimora/outputs/geographic/crime_coordinates.csv
Total exported rows: 52,119
Columns: ['uid', 'city', 'state', 'lat', 'lon', 'reported_date_clean', 'reported_year', 'reported_month', 'reported_day_of_week', 'reported_is_weekend', 'is_solved', 'disposition', 'latitude', 'longitude']
Privacy verification: Zero PII fields present in export.


## 4. Distance Calculation

Implement the Great-Circle Haversine spherical distance metric:
$$d = 2 R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)}\right)$$
where $R = 6371.0$ km. Pairwise distance matrices are computed with vectorized broadcasting for maximum computational efficiency.

In [4]:
import numpy as np
from src.geo_utils import haversine_distance, pairwise_haversine_matrix

# Reference geodesic distance check: Baltimore City Hall to Washington DC Capitol
balt_lat, balt_lon = 39.2904, -76.6122
dc_lat, dc_lon = 38.8899, -77.0091

dist_km = haversine_distance(balt_lat, balt_lon, dc_lat, dc_lon)
print(f"Calculated Haversine Distance (Baltimore to DC): {dist_km:.2f} km (Expected ~56.5 km)")

# Pairwise test on sample coordinate points
sample_pts = clean_coords_df[clean_coords_df["city"] == "Baltimore"][["lat", "lon"]].head(5).to_numpy()
dist_mat = pairwise_haversine_matrix(sample_pts, sample_pts)
print("\nPairwise 5x5 Haversine Distance Matrix (km):")
print(np.round(dist_mat, 2))

Calculated Haversine Distance (Baltimore to DC): 56.18 km (Expected ~56.5 km)

Pairwise 5x5 Haversine Distance Matrix (km):
[[ 0.    2.02  8.83 10.22  7.44]
 [ 2.02  0.   10.29 11.71  8.98]
 [ 8.83 10.29  0.   13.95  1.46]
 [10.22 11.71 13.95  0.   12.72]
 [ 7.44  8.98  1.46 12.72  0.  ]]


## 5. LWR / Distance-Weighted Method

### Methodological Comparison: Spatial Point Intensity vs. Locally Weighted Regression
- **Spatial Intensity Estimation (Geographic Profiling):** Crime occurrences form discrete spatial point processes. We estimate the continuous event occurrence intensity $\lambda(x) = \frac{1}{N h^2} \sum_{i=1}^N K\left(\frac{d(x, x_i)}{h}\right)$ using smooth spatial kernels (Gaussian, Epanechnikov, Tri-cube, Exponential).
- **Locally Weighted Regression (LWR / LOESS):** In contrast, standard LWR requires a continuous response variable $Y \in \mathbb{R}$. We formulate the closed-form local linear solver $\hat{\beta}(x_0) = (X^T W(x_0) X + \lambda I)^{-1} X^T W(x_0) y$ with diagonal kernel weight matrix $W(x_0)$.
Both modules are implemented in `src/lwr_profiler.py`.

In [5]:
from src.lwr_profiler import (
    SpatialIntensityProfiler, LocallyWeightedRegressor,
    estimate_silverman_bandwidth, get_kernel_function
)

# 1. Bandwidth Estimation
balt_df = clean_coords_df[clean_coords_df["city"] == "Baltimore"].copy()
balt_coords = balt_df[["lat", "lon"]].to_numpy()
silverman_h = estimate_silverman_bandwidth(balt_coords)
print(f"Baltimore Incidents: {len(balt_coords):,}")
print(f"Silverman Rule Bandwidth: {silverman_h:.4f} km")

# 2. Kernel Weighting Verification
u_vals = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
for k_name in ["gaussian", "epanechnikov", "tricube", "exponential"]:
    k_fn = get_kernel_function(k_name)
    w = k_fn(u_vals)
    print(f"Kernel '{k_name:<12}': u=[0, 0.5, 1, 1.5, 2] -> w={np.round(w, 3)}")

# 3. LWR demonstration on continuous response (victim age surface)
lwr_model = LocallyWeightedRegressor(bandwidth_km=2.0, kernel="tricube")
lwr_model.fit(balt_coords[:300], df_hom.loc[balt_df.index[:300], "victim_age_clean"].fillna(30).to_numpy())
query_pt = np.array([39.29, -76.61])
local_age_pred = lwr_model.predict_point(query_pt)
print(f"\nLWR Estimated Local Victim Age at {query_pt}: {local_age_pred:.1f} years")

Baltimore Incidents: 2,827
Silverman Rule Bandwidth: 0.8574 km
Kernel 'gaussian    ': u=[0, 0.5, 1, 1.5, 2] -> w=[0.399 0.352 0.242 0.13  0.054]
Kernel 'epanechnikov': u=[0, 0.5, 1, 1.5, 2] -> w=[0.75  0.562 0.    0.    0.   ]
Kernel 'tricube     ': u=[0, 0.5, 1, 1.5, 2] -> w=[1.   0.67 0.   0.   0.  ]
Kernel 'exponential ': u=[0, 0.5, 1, 1.5, 2] -> w=[1.    0.607 0.368 0.223 0.135]

LWR Estimated Local Victim Age at [ 39.29 -76.61]: 32.6 years


## 6. Spatial Grid Generation

Construct a regular 2D evaluation mesh grid over the focal metropolitan extent (Baltimore, MD, spanning $2,827$ geocoded homicides in a compact bounding box). A $50 \times 50$ grid ($2,500$ evaluation nodes) provides fine spatial granularity (~300m resolution) without unnecessary computational overhead.

In [6]:
from src.geo_utils import calculate_geographic_bounds, generate_spatial_grid

bounds = calculate_geographic_bounds(balt_df, lat_col="lat", lon_col="lon", buffer_ratio=0.06)
print(f"Bounding Box: Lat [{bounds['min_lat']:.4f}, {bounds['max_lat']:.4f}], Lon [{bounds['min_lon']:.4f}, {bounds['max_lon']:.4f}]")
print(f"Geographic Span: Lat {bounds['span_lat_km']:.1f} km, Lon {bounds['span_lon_km']:.1f} km")

grid_df = generate_spatial_grid(
    min_lat=bounds["min_lat"],
    max_lat=bounds["max_lat"],
    min_lon=bounds["min_lon"],
    max_lon=bounds["max_lon"],
    resolution_lat=50,
    resolution_lon=50,
)
print(f"Generated Grid: {len(grid_df)} cells (50x50 mesh).")
print(grid_df.head(3))

Bounding Box: Lat [39.2137, 39.3809], Lon [-76.7218, -76.5201]
Geographic Span: Lat 16.6 km, Lon 15.5 km
Generated Grid: 2500 cells (50x50 mesh).
   grid_id  lat_idx  lon_idx   latitude  longitude
0        0        0        0  39.213746 -76.721827
1        1        0        1  39.213746 -76.717711
2        2        0        2  39.213746 -76.713595


## 7. Geographic Intensity Estimation

Evaluate distance-weighted spatial intensity across all grid evaluation nodes using the fitted `SpatialIntensityProfiler`. The raw intensity is normalized to $[0.0, 1.0]$ and relative probability density summing to $1.0$, then serialized to `outputs/geographic/geographic_grid.csv`.

In [7]:
from src.lwr_profiler import save_geographic_grid

profiler = SpatialIntensityProfiler(bandwidth_km=silverman_h, kernel="gaussian")
profiler.fit(balt_coords)

evaluated_grid = profiler.evaluate_grid(grid_df, lat_col="latitude", lon_col="longitude")

grid_out_path = GEOGRAPHIC_OUTPUTS_DIR / "geographic_grid.csv"
save_geographic_grid(evaluated_grid, grid_out_path)

print(f"Evaluated grid shape: {evaluated_grid.shape}")
print(f"Raw Intensity: min={evaluated_grid['raw_intensity'].min():.2e}, max={evaluated_grid['raw_intensity'].max():.2e}")
print(f"Normalized Intensity: min={evaluated_grid['normalized_intensity'].min():.4f}, max={evaluated_grid['normalized_intensity'].max():.4f}")
print(f"Saved to: {grid_out_path}")

Evaluated grid shape: (2500, 9)
Raw Intensity: min=1.50e-18, max=4.88e-02
Normalized Intensity: min=0.0000, max=1.0000
Saved to: /home/Dharsit/ML-Project/Crimora/outputs/geographic/geographic_grid.csv


## 8. Hotspot Identification

Identify peak crime concentration areas from the continuous intensity surface. Grid cells exceeding the 95th percentile are clustered using spatial non-maximum suppression (minimum 1.0 km separation) to isolate distinct analytical hotspot centroids.

> [!IMPORTANT]
> Analytical hotspots denote historical geographic clusters of reported crime activity for operational decision support. They **do not** denote offender residence or identify individual suspects.

In [8]:
from src.lwr_profiler import save_hotspot_summary

hotspots_df = profiler.identify_hotspots(
    evaluated_grid=evaluated_grid,
    threshold_percentile=95.0,
    top_n=10,
    min_separation_km=1.0,
    lat_col="latitude",
    lon_col="longitude",
)
hotspots_df["city"] = "Baltimore"

hotspots_out_path = GEOGRAPHIC_OUTPUTS_DIR / "hotspot_summary.csv"
save_hotspot_summary(hotspots_df, hotspots_out_path)

print("=== IDENTIFIED ANALYTICAL HOTSPOT CENTROIDS (BALTIMORE) ===")
print(hotspots_df.to_string(index=False))
print(f"\nSaved hotspot summary to: {hotspots_out_path}")

=== IDENTIFIED ANALYTICAL HOTSPOT CENTROIDS (BALTIMORE) ===
 hotspot_id  latitude  longitude  estimated_intensity  normalized_intensity  relative_rank percentile_tier      city
          1 39.302423 -76.643620             0.048834                1.0000              1        >=95.0th Baltimore
          2 39.292191 -76.643620             0.047705                0.9769              2        >=95.0th Baltimore
          3 39.309244 -76.594226             0.047138                0.9653              3        >=95.0th Baltimore
          4 39.302423 -76.585994             0.043479                0.8903              4        >=95.0th Baltimore
          5 39.316066 -76.602458             0.043022                0.8810              5        >=95.0th Baltimore
          6 39.309244 -76.651852             0.040623                0.8319              6        >=95.0th Baltimore
          7 39.299012 -76.655968             0.040460                0.8285              7        >=95.0th Baltimore
    

## 9. Geographic Visualizations

Verify and display the four publication-quality static analytical plots:
1. `incident_distribution.png`: Point pattern distribution with density contours.
2. `intensity_surface.png`: Continuous normalized spatial intensity surface.
3. `hotspot_analysis.png`: Hotspot centroids and 95th percentile concentration boundary.
4. `incidents_vs_intensity.png`: Side-by-side comparison of raw incidents vs estimated activity area.

In [9]:
from src.run_geographic_pipeline import generate_geographic_visualizations
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

plot_paths = generate_geographic_visualizations(
    urban_df=balt_df,
    evaluated_grid=evaluated_grid,
    hotspots_df=hotspots_df,
    bounds=bounds,
    city_name="Baltimore",
    output_dir=GEOGRAPHIC_OUTPUTS_DIR,
)

print(f"Generated {len(plot_paths)} visualization figures:")
for p in plot_paths:
    print(f"  - {p.name} ({p.stat().st_size:,} bytes)")

# Display side-by-side comparison plot
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
img = mpimg.imread(GEOGRAPHIC_OUTPUTS_DIR / "incidents_vs_intensity.png")
ax.imshow(img)
ax.axis("off")
ax.set_title("Publication Figure: Raw Incidents vs. Estimated Activity-Area Surface", pad=10)
plt.show()

Generated 4 visualization figures:
  - incident_distribution.png (1,157,009 bytes)
  - intensity_surface.png (327,651 bytes)
  - hotspot_analysis.png (401,367 bytes)
  - incidents_vs_intensity.png (1,085,141 bytes)


## 10. Interactive Folium Map

Generate an interactive Leaflet/Folium spatial map (`outputs/geographic/geographic_profile.html`) featuring:
- Standard OpenStreetMap and CartoDB base tiles
- Clustered incident marker layer with sanitized case outcome popups
- Continuous heatmap intensity layer (`folium.plugins.HeatMap`)
- Distinct analytical hotspot markers with ranking and score badges
- Folium `LayerControl` for interactive toggling

In [10]:
from src.run_geographic_pipeline import generate_interactive_folium_map

folium_html_path = GEOGRAPHIC_OUTPUTS_DIR / "geographic_profile.html"
generate_interactive_folium_map(
    urban_df=balt_df,
    evaluated_grid=evaluated_grid,
    hotspots_df=hotspots_df,
    city_name="Baltimore",
    output_html=folium_html_path,
)

print(f"Interactive Folium map generated at: {folium_html_path}")
print(f"File size: {folium_html_path.stat().st_size:,} bytes")
print("Verified: Map is self-contained and ready for Streamlit embedding via st.components.v1.html().")

Interactive Folium map generated at: /home/Dharsit/ML-Project/Crimora/outputs/geographic/geographic_profile.html
File size: 2,168,983 bytes
Verified: Map is self-contained and ready for Streamlit embedding via st.components.v1.html().


## 11. Validation

Execute comprehensive programmatic validation checks covering all requirements:
1. Coordinate validity $[-90, 90]$ and $[-180, 180]$
2. Safe segregation of missing coordinates (no zero replacement)
3. Non-negative and finite intensity values
4. Monotonic normalization $[0.0, 1.0]$
5. Reproducible hotspot ranking
6. Output file presence and schema integrity
7. Raw dataset immutability

In [11]:
from src.config import RAW_DATA_DIR

# 1. Coordinate ranges
assert (-90.0 <= clean_coords_df["latitude"]).all() and (clean_coords_df["latitude"] <= 90.0).all()
assert (-180.0 <= clean_coords_df["longitude"]).all() and (clean_coords_df["longitude"] <= 180.0).all()

# 2. No zeros in place of missing coords
assert not ((clean_coords_df["latitude"] == 0.0) & (clean_coords_df["longitude"] == 0.0)).any()

# 3. Finite, non-negative intensity
assert np.all(np.isfinite(evaluated_grid["raw_intensity"]))
assert np.all(evaluated_grid["raw_intensity"] >= 0.0)
assert 0.0 <= evaluated_grid["normalized_intensity"].min() <= 1e-6
assert 0.9999 <= evaluated_grid["normalized_intensity"].max() <= 1.0001

# 4. Hotspots integrity
assert len(hotspots_df) > 0
assert (hotspots_df["relative_rank"] == np.arange(1, len(hotspots_df) + 1)).all()

# 5. Output file reloadability
reloaded_coords = pd.read_csv(coords_export_path)
reloaded_grid = pd.read_csv(grid_out_path)
reloaded_hs = pd.read_csv(hotspots_out_path)
assert len(reloaded_coords) == len(clean_coords_df)
assert len(reloaded_grid) == len(evaluated_grid)
assert len(reloaded_hs) == len(hotspots_df)

# 6. Raw data immutability
for raw_name in ["homicide-data.csv", "dstrIPC_1_2014.csv", "TN-murder-2023.csv", "TN-2020-2022-total.csv"]:
    assert (RAW_DATA_DIR / raw_name).exists(), f"Raw file {raw_name} missing"

print("ALL DAY 4 VALIDATION SUITES PASSED (100% SUCCESS)")

ALL DAY 4 VALIDATION SUITES PASSED (100% SUCCESS)


## 12. Limitations

### Methodological and Operational Limitations
1. **Reporting and Selection Bias:** Incident records reflect police-reported crimes and clearance classifications; areas with higher patrol intensity or reporting rates may display artificially elevated crime concentration.
2. **Modifiable Areal Unit Problem (MAUP):** Spatial intensity estimations and hotspot boundaries vary with the chosen evaluation grid resolution and coordinate discretization.
3. **Bandwidth Sensitivity:** The choice of kernel bandwidth $h$ influences the degree of smoothing: smaller bandwidths identify micro-hotspots (individual street blocks) with high variance, while larger bandwidths identify broad regional activity corridors.
4. **Point Process Assumptions:** Standard distance-weighted profiling assumes stationary spatial processes across Euclidean/spherical distances, abstracting away urban physical barriers (waterways, railways, highways).
5. **Ethical Decision-Support Boundary:** Results represent **historical spatial patterns of reported events** for administrative resource allocation. They must never be interpreted as individual risk scores, predictive offender profiling, or evidence of guilt.